<a href="https://colab.research.google.com/github/Holanpasaribu12/UAS/blob/main/ToxicComentDeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# TOXIC COMMENT CLASSIFICATION - INDOBERT
# FULL GOOGLE COLAB CODE (FIX ERROR VERSION)
# =========================================================

# =========================================================
# 1. INSTALL LIBRARY
# =========================================================

!pip install -U transformers
!pip install -U accelerate
!pip install -U datasets
!pip install -U scikit-learn
!pip install -U pandas

# =========================================================
# 2. IMPORT LIBRARY
# =========================================================

import pandas as pd
import numpy as np
import torch
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# =========================================================
# 3. UPLOAD DATASET
# =========================================================

from google.colab import files

uploaded = files.upload()

# upload:
# - combined_dataset.csv
# - kamus_singkatan.csv
# - stopwordsID.csv

# =========================================================
# 4. LOAD DATASET
# =========================================================

df = pd.read_csv("combined_dataset.csv")

print("Jumlah Data:", len(df))

print(df.head())

# =========================================================
# 5. AMBIL KOLOM
# =========================================================

df = df[['clean_text', 'encoded_label']]

df.columns = ['text', 'label']

print(df.head())

# =========================================================
# 6. HAPUS DATA KOSONG
# =========================================================

df = df.dropna()

df['label'] = df['label'].astype(int)

# =========================================================
# 7. CLEANING TEXT
# =========================================================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#\w+", "", text)

    text = re.sub(r"\d+", "", text)

    text = re.sub(r"[^\w\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

df['text'] = df['text'].apply(clean_text)

print("\nData Setelah Cleaning:\n")
print(df.head())

# =========================================================
# 8. SPLIT DATA
# =========================================================

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

print("\nJumlah Training:", len(train_texts))
print("Jumlah Validation:", len(val_texts))

# =========================================================
# 9. LOAD TOKENIZER
# =========================================================

model_name = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=128
)

# =========================================================
# 10. DATASET CLASS
# =========================================================

class ToxicDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):

        return len(self.labels)

train_dataset = ToxicDataset(train_encodings, train_labels)

val_dataset = ToxicDataset(val_encodings, val_labels)

# =========================================================
# 11. LOAD MODEL
# =========================================================

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# =========================================================
# 12. METRIC
# =========================================================

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc
    }

# =========================================================
# 13. TRAINING ARGUMENT
# =========================================================

training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    logging_steps=10,

    save_steps=100,

    do_train=True,

    do_eval=True
)

# =========================================================
# 14. TRAINER
# =========================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

# =========================================================
# 15. TRAIN MODEL
# =========================================================

print("\nTRAINING MODEL...\n")

trainer.train()

# =========================================================
# 16. EVALUASI MODEL
# =========================================================

print("\nEVALUASI MODEL...\n")

predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(val_labels, preds))

# =========================================================
# 17. TEST MANUAL
# =========================================================

print("\n========================================")
print("TEST TOXIC COMMENT")
print("ketik 'exit' untuk berhenti")
print("========================================")

while True:

    text = input("\nMasukkan komentar: ")

    if text.lower() == "exit":
        break

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    outputs = model(**inputs)

    pred = torch.argmax(outputs.logits).item()

    # dataset kamu:
    # 0 = bullying/toxic
    # 1 = non-bullying

    if pred == 0:
        print("HASIL: Toxic Comment")
    else:
        print("HASIL: Non-Toxic")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 29.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 23.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 79.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
     ━━━━━━━━

Saving combined_dataset.csv to combined_dataset.csv
Saving kamus_singkatan.csv to kamus_singkatan.csv
Saving stopwordsID.csv to stopwordsID.csv
Jumlah Data: 2066
   Unnamed: 0         Label  \
0           0  Non-bullying   
1           1  Non-bullying   
2           2      Bullying   
3           3  Non-bullying   
4           4  Non-bullying   

                                          clean_text  \
0       kaka tidur yaa sudah pagi tidak boleh capek2   
1                    makan nasi padang saja badannya   
2                         suka cukur jembut manggung   
3  hai kak isyana ngefans sekali kak isyana suka ...   
4             manusia bidadari sih herann deh cantik   

                                              String  encoded_label  
0        "Kaka tidur yaa, udah pagi, gaboleh capek2"            1.0  
1            "makan nasi padang aja begini badannya"            1.0  
2  "yang aku suka dari dia adalah selalu cukur je...            0.0  
3  "Hai kak Isyana aku ngefans ban

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


EVALUASI MODEL...



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.94      0.93      0.94       219
           1       0.92      0.94      0.93       195

    accuracy                           0.93       414
   macro avg       0.93      0.93      0.93       414
weighted avg       0.93      0.93      0.93       414


TEST TOXIC COMMENT
ketik 'exit' untuk berhenti
